**Instructions:**

- Do not write your name on the assignment. Be careful about any warnings that might display some file path with your name included.
- Receiving help from classmates and/or ideas from Generative AI is allowed. **However, you must submit your own original work.**
- For questions that require coding, you need to write the relevant code and display its output. Your output should either be the direct answer to the question or clearly display the answer in it.
- For questions that require a written answer (sometimes along with the code), you need to put your answer in a Markdown cell. Writing the answer as a comment or as a print line is not acceptable.
- You need to render this file as HTML using Quarto and submit the HTML file. **Please note that this is a requirement and not optional.** A submission cannot be graded until it is properly rendered.

Import all the libraries and tools you need below.

In [29]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.metrics import hamming_loss, accuracy_score

### 1)

Read the data from **X_train.csv**, **y_train.csv**, **X_test.csv**, and **y_test.csv**. The features (X's) are rhythmic and timbre features extracted from a number of songs. The responses (y's) are the emotion class labels for each song. **(2.5 points)**

In [3]:
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

### 2)

Print the first five rows of either `y_train` or `y_test`. You should observe that each observation has multiple class labels and it is possible for an observation to have multiple Class 1 values.

**(2.5 points)**

In [5]:
y_test.head(5)

,"('amazed-suprised', ['0', '1'])","('happy-pleased', ['0', '1'])","('relaxing-calm', ['0', '1'])","('quiet-still', ['0', '1'])","('sad-lonely', ['0', '1'])","('angry-aggresive', ['0', '1'])"
0,0,0,1,1,1,0
1,1,0,0,0,0,1
2,0,0,0,0,1,0
3,0,1,1,0,0,0
4,0,0,0,0,1,0


### 3)

Create a [Random Forest Classifier](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestClassifier.html). Use 500 trees, so that its variance is reduced adequately. Leave all the other hyperparameters default; tuning their values does not change the results substantially. Use `random_state=2` for reproducibility. **(5 points)**

In [12]:
random_forest = RandomForestClassifier(n_estimators=500, random_state=2)

### 4)

Train the Random Forest on the multi-label data using the **Binary Relevance** approach. You need to check the [scikit-learn documentation](https://scikit-learn.org/stable/api/sklearn.multioutput.html) for the correct object and its usage.

Evaluate the multi-label classifier on the test data, using [Hamming Loss](https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.hamming_loss.html).

**(15 points)**

In [17]:
binary_classifier = MultiOutputClassifier(random_forest, n_jobs=-1)
binary_classifier.fit(X_train, y_train)

y_pred = binary_classifier.predict(X_test)

loss = hamming_loss(y_test, y_pred)

print(f"Hamming Loss: {round(loss, 3)}")

Hamming Loss: 0.196


### 5)

What does the Hamming Loss represent in terms of what the model predicts right/wrong? **(10 points)**

The Hamming Loss is a metric used to evaluate perfromance of multi-label classification models. The 0.196 Hamming Loss means that the model incorrectly predicted 19.6% of the labels and predicted 80.4% of the labels correctly. In other words, the model is wrong about 19.6% of the labels and right about 80.4% of the labels.

### 6)

Train the Random Forest on the multi-label data using the **Classifier Chain** approach. Keep all the inputs (other than the base model) default. You may need to refer back to the scikit-learn documentation for the correct object.

Evaluate the multi-label classifier on the test data, using Hamming Loss. You should see the same performance as Question 4.

**(15 points)**

In [18]:
chain_classifier = ClassifierChain(random_forest)
chain_classifier.fit(X_train, y_train)
y_pred = chain_classifier.predict(X_test)
loss = hamming_loss(y_test, y_pred)

print(f"Hamming Loss: {round(loss, 3)}")

Hamming Loss: 0.196


### 7)

Using the scikit-learn documentation, answer the following about the multi-label model in Question 6:

- Are you using the true or the predicted classes from the previous classifier(s) as the features of the next classifier in the chain? (Please make sure you read the documentation.)
- What is the order of class variables that you use for the chain?

**(10 points)**

1) In training, the Classifier Chain uses the true labels from previous classifiers as features for the next classifier because by default cv=None. But when testings, it uses the predicted labels because the true labels are not available.

2) Since I used the defauld order, which is None, the chain follows the original order of the the class variables in y_train, starting from the first column and going all the way to the last column.

### 8)

Repeat Question 6, only with `cv=5` as another input to the Classifier Chain. What does this change about the multi-label model?

You should see a slightly lower performance. Why is this a more realistic evaluation of the model?

**(10 points)**

In [19]:
chain_classifier = ClassifierChain(random_forest, cv=5)
chain_classifier.fit(X_train, y_train)
y_pred = chain_classifier.predict(X_test)
loss = hamming_loss(y_test, y_pred)

print(f"Hamming Loss: {round(loss, 3)}")

Hamming Loss: 0.197


Using cv=5 changes the Classifier Chain by using cross-validated predictions from previous classifiers instead of the true labels during training. It makes the model more realistic because when predicting new data, we do not know the true labels and have to rely on predictions from previous classifiers.

The Hamming Loss increased from 0.196 to 0.197, so the model performed slightly worse. It makes sense because the predicted labels can contain errors, unlike the true labels. But still, this approach is better at representing how the model will actually work on new data.

### 9)

Run the given cell below. It calculates and prints the Variation Inflation Factor (VIF) of each class variable.

VIF is a way to aggregate the multi-collinearity each variable has with all the other variables. The higher the VIF value of a variable is, the higher its total correlation is with all the other variables. **Note that having a correlation/multi-collinearity with other variables means carrying some information about other variables.**

**(0 points)**

In [20]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
y = add_constant(y_train)
vif_data = pd.DataFrame()
vif_data["feature"] = y.columns

for i in range(len(y.columns)):
    vif_data.loc[i,'VIF'] = variance_inflation_factor(y.values, i)

print(vif_data)

                           feature        VIF
0                            const  10.591750
1  ('amazed-suprised', ['0', '1'])   1.434148
2    ('happy-pleased', ['0', '1'])   1.676727
3    ('relaxing-calm', ['0', '1'])   1.782708
4      ('quiet-still', ['0', '1'])   1.630620
5       ('sad-lonely', ['0', '1'])   1.623160
6  ('angry-aggresive', ['0', '1'])   2.126801


### 10)

Using the output of Question 9, repeat Question 8, only this time with the **most informative** order of the class variables. (Python starts counting from zero.)

You should see the best model performance in this assignment. Why is this the case?

**(15 points)**

In [28]:
vif_data = vif_data.sort_values('VIF', ascending=False)

order = (vif_data.iloc[1:].index - 1).tolist()

chain_classifier_vif = ClassifierChain(
    random_forest,
    cv=5,
    order=order
)

chain_classifier_vif.fit(X_train, y_train)

y_pred_vif = chain_classifier_vif.predict(X_test)

loss_vif = hamming_loss(y_test, y_pred_vif)

print(f"Hamming Loss: {loss_vif:.3f}")

Hamming Loss: 0.194


By reordering the class variables based on their VIF values, starting with the highest VIF, allows the labels that share more information with other labels to be predicted earlier in the chain, helping later classifiers. The Hamming Loss decreased from 0.197 to 0.194, making it the best performing model. Using more informative labels earlier in the chain helps the model improve predictions for the following labels because they can user information from previous predicstions.

### 11)

Finally, using the predictions of the best Classifier Chain (from Question 10), calculate and print the test accuracy of the model **for each emotion**. Which emotions are predicted the most and least accurately?

**(15 points)**

In [31]:
accuracies = {}

for i, emotion in enumerate(y_test.columns):
    name = emotion[0] if isinstance(emotion, tuple) else emotion
    accuracies[name] = accuracy_score(y_test.iloc[:, i], y_pred_vif[:, i])

accuracies = pd.Series(accuracies).sort_values(ascending=False)

print(accuracies)

('quiet-still', ['0', '1'])        0.866337
('angry-aggresive', ['0', '1'])    0.846535
('amazed-suprised', ['0', '1'])    0.811881
('sad-lonely', ['0', '1'])         0.792079
('happy-pleased', ['0', '1'])      0.782178
('relaxing-calm', ['0', '1'])      0.737624
dtype: float64


Most accurate: quiet-still

Least accurate: relazing-calm